In [37]:
import requests
from bs4 import BeautifulSoup

Response = requests.get("https://www.honda-indonesia.com/dealers/")
print(Response.status_code)
# print(Response.text)

200


In [38]:
soup = BeautifulSoup(Response.text, 'html.parser')
table_blocks = soup.find_all('div', class_='tw-space-y-4')
print("Number of countries found: ", len(table_blocks))

Number of countries found:  194


In [39]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

# --- Request ---
url = "https://www.honda-indonesia.com/dealers/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()
print(f"Status: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

# --- Cari semua konten provinsi (div dengan x-show="open === X") ---
province_content_divs = soup.find_all('div', attrs={'x-show': lambda x: x and 'open ===' in x})

print(f"Jumlah provinsi terdeteksi: {len(province_content_divs)}")

result = []

for content_div in province_content_divs:
    try:
        # 1. Ambil ID provinsi dari x-show
        x_show = content_div.get('x-show')
        province_id = re.search(r'open === (\d+)', x_show).group(1)

        # 2. Cari button provinsi (sebelum content_div)
        button = content_div.find_previous('button')
        if not button:
            continue

        span = button.find('span', class_='tw-font-bold')
        province_name = span.get_text(strip=True) if span else "Unknown"

        # 3. Cari semua blok dealer di dalam content_div
        dealer_blocks = content_div.find_all('div', class_='tw-space-y-4')

        for block in dealer_blocks:
            try:
                # Nama Dealer
                name_tag = block.find('a', href=lambda x: x and '/dealers/' in x)
                dealer = name_tag.get_text(strip=True) if name_tag else "N/A"

                # Alamat
                alamat_tag = block.find('div', class_='tw-text-gray-500')
                alamat = alamat_tag.get_text(strip=True) if alamat_tag else "Tidak tersedia"

                # Koordinat
                direction_tag = None
                for a in block.find_all('a'):
                    if a.get_text(strip=True) == "Direction":
                        direction_tag = a
                        break

                lat, long = "N/A", "N/A"
                if direction_tag and direction_tag.get('href'):
                    href = direction_tag['href']
                    match = re.search(r'q=([-\d.]+),([-\d.]+)', href)
                    if match:
                        lat, long = match.group(1), match.group(2)

                # Simpan
                result.append({
                    'Provinsi': province_name,
                    'Dealer': dealer,
                    'Alamat': alamat,
                    'Latitude': lat,
                    'Longitude': long
                })

            except Exception as e:
                print(f"Error parsing dealer block: {e}")
                continue

    except Exception as e:
        print(f"Error parsing province: {e}")
        continue

# --- Output ---
print("\n" + "="*100)
print("CONTOH 5 DEALER PERTAMA".center(100))
print("="*100)
for item in result[:5]:
    print(f"Provinsi   : {item['Provinsi']}")
    print(f"Dealer     : {item['Dealer']}")
    print(f"Alamat     : {item['Alamat']}")
    print(f"Lat/Long   : {item['Latitude']}, {item['Longitude']}")
    print("-" * 100)


Status: 200
Jumlah provinsi terdeteksi: 32

                                      CONTOH 5 DEALER PERTAMA                                       
Provinsi   : Bali
Dealer     : Honda Bintang Tabanan
Alamat     : 
Lat/Long   : N/A, N/A
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Cokroaminoto
Alamat     : Jl. Cokroaminoto No. 168
Lat/Long   : N/A, N/A
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Denpasar Agung
Alamat     : Jl. Hayam Wuruk No. 40
Lat/Long   : -8.65731, 115.22685
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Dewata Motor
Alamat     : Jl. Imam Bonjol No. 104
Lat/Long   : -8.66767, 115.20567
----------------------------------------------------------------------------------------------------
Provinsi   

In [40]:
# --- Simpan ke Excel ---
import pandas as pd

df = pd.DataFrame(result)

# Add headers
df.columns = ['Provinsi', 'Dealer', 'Alamat', 'Latitude', 'Longitude']

# Save to 
df

,Provinsi,Dealer,Alamat,Latitude,Longitude
0,Bali,Honda Bintang Tabanan,,N/A,N/A
1,Bali,Honda Cokroaminoto,Jl. Cokroaminoto No. 168,N/A,N/A
2,Bali,Honda Denpasar Agung,Jl. Hayam Wuruk No. 40,-8.65731,115.22685
3,Bali,Honda Dewata Motor,Jl. Imam Bonjol No. 104,-8.66767,115.20567
4,Banten,Honda Arta Cikupa,"Jl. Raya Serang KM 14, Cikupa",-6.22295,106.529
...,...,...,...,...,...
187,Sumatera Utara,Honda Arista Siantar,,N/A,N/A
188,Sumatera Utara,Honda Arista SM Raja,Jl. Sisingamangaraja Km. 5.5 No. 2,3.54615,98.69831
189,Sumatera Utara,Honda IDK 1,Jl.Glugur By Pass No. 85,3.60232,98.66879
190,Sumatera Utara,Honda IDK 2,Jl. Sei Batang Hari No. 22-24,3.58511,98.65167


In [41]:
# --- Scraping Koordinat dengan Selenium untuk Dealer yang Belum Terisi ---
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time

# Pastikan DataFrame df sudah ada
if 'df' not in locals() and 'df' not in globals():
    if 'result' in locals() or 'result' in globals():
        df = pd.DataFrame(result)
        df.columns = ['Provinsi', 'Dealer', 'Alamat', 'Latitude', 'Longitude']
    else:
        raise ValueError("DataFrame 'df' tidak ditemukan. Jalankan Cell 2 dan Cell 3 terlebih dahulu.")

# Filter dealer yang belum punya koordinat
mask = (df['Latitude'] == 'N/A') | (df['Latitude'].isna()) | (df['Longitude'] == 'N/A') | (df['Longitude'].isna())
dealers_to_scrape = df[mask].copy()

print(f"Total dealer: {len(df)}")
print(f"Perlu di-scrape: {len(dealers_to_scrape)} | Sudah ada koordinat: {len(df[~mask])}")
if len(dealers_to_scrape) > 0:
    print(f"\nContoh dealer yang akan di-scrape:")
    print(dealers_to_scrape[['Dealer', 'Alamat', 'Latitude', 'Longitude']].head())

Total dealer: 192
Perlu di-scrape: 50 | Sudah ada koordinat: 142

Contoh dealer yang akan di-scrape:
                   Dealer                                             Alamat  \
0   Honda Bintang Tabanan                                                      
1      Honda Cokroaminoto                           Jl. Cokroaminoto No. 168   
5      Honda Auto Cilegon                                                      
17  Honda Anugerah Bantul  Jl. Ringroad Barat, Tamantirto, Kasihan, Kabup...   
29     Honda Maju Pd Gede                                                      

   Latitude Longitude  
0       N/A       N/A  
1       N/A       N/A  
5       N/A       N/A  
17      N/A       N/A  
29      N/A       N/A  


In [42]:
# --- Setup Selenium WebDriver ---
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=chrome_options)
print("WebDriver Chrome sudah diinisialisasi")


WebDriver Chrome sudah diinisialisasi


In [43]:
# --- Fungsi untuk extract koordinat dari URL Google Maps ---
def extract_coordinates_from_url(url):
    """Extract koordinat dari URL Google Maps dengan berbagai format"""
    try:
        patterns = [
            r'@(-?\d+\.?\d*),(-?\d+\.?\d*)',      # Format @lat,long
            r'[?&]q=(-?\d+\.?\d*),(-?\d+\.?\d*)', # Format q=lat,long
            r'/dir/(-?\d+\.?\d*),(-?\d+\.?\d*)',   # Format /dir/lat,long
            r'center=(-?\d+\.?\d*),(-?\d+\.?\d*)'  # Format center=lat,long
        ]
        
        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                lat, lon = match.group(1), match.group(2)
                if -90 <= float(lat) <= 90 and -180 <= float(lon) <= 180:
                    return lat, lon
    except:
        pass
    return None, None


In [44]:
# --- Fungsi untuk scraping koordinat dari Google Maps ---
def scrape_coordinates(dealer_name, alamat=""):
    """Search dealer di Google Maps dan extract koordinat dari URL"""
    try:
        # Buat search query
        query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
        maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        
        driver.get(maps_url)
        time.sleep(4)
        
        # Extract koordinat dari URL
        current_url = driver.current_url
        lat, lon = extract_coordinates_from_url(current_url)
        
        if lat and lon:
            return lat, lon
        
        # Fallback: klik hasil pertama jika belum dapat koordinat
        try:
            first_result = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "div[role='article']:first-of-type"))
            )
            first_result.click()
            time.sleep(3)
            current_url = driver.current_url
            lat, lon = extract_coordinates_from_url(current_url)
            if lat and lon:
                return lat, lon
        except:
            pass
        
        return None, None
    except Exception as e:
        print(f"Error scraping {dealer_name}: {str(e)}")
        return None, None


In [45]:
# --- Loop untuk scraping semua dealer yang belum punya koordinat ---
if len(dealers_to_scrape) > 0:
    success_count = failed_count = 0
    total = len(dealers_to_scrape)
    
    print(f"\n{'='*80}")
    print(f"MULAI SCRAPING {total} DEALER")
    print(f"{'='*80}\n")
    
    for i, (idx, row) in enumerate(dealers_to_scrape.iterrows(), 1):
        dealer_name = row['Dealer']
        alamat = row['Alamat'] if pd.notna(row['Alamat']) else ""
        
        print(f"[{i}/{total}] {dealer_name}")
        lat, lon = scrape_coordinates(dealer_name, alamat)
        
        if lat and lon:
            df.at[idx, 'Latitude'] = lat
            df.at[idx, 'Longitude'] = lon
            success_count += 1
            print(f"  ✓ Lat={lat}, Long={lon}")
        else:
            failed_count += 1
            print(f"  ✗ Gagal")
        
        time.sleep(2)
        
        if i % 10 == 0:
            df.to_excel('honda_dealers_indonesia.xlsx', index=False)
            print(f"\n💾 Progress: {success_count} sukses, {failed_count} gagal\n")
    
    print(f"\n{'='*80}")
    print(f"SCRAPING SELESAI!")
    print(f"Sukses: {success_count} | Gagal: {failed_count} | Rate: {(success_count/total*100):.1f}%")
    print(f"{'='*80}")
    
    df.to_excel('honda_dealers_indonesia.xlsx', index=False)
    mask_final = (df['Latitude'] == 'N/A') | (df['Latitude'].isna()) | (df['Longitude'] == 'N/A') | (df['Longitude'].isna())
    print(f"\n📊 Dealer dengan koordinat: {len(df[~mask_final])} | Tanpa koordinat: {len(df[mask_final])}")
else:
    print("Tidak ada dealer yang perlu di-scrape")



MULAI SCRAPING 50 DEALER

[1/50] Honda Bintang Tabanan
  ✓ Lat=-8.5530323, Long=115.1362066
[2/50] Honda Cokroaminoto
  ✗ Gagal
[3/50] Honda Auto Cilegon
  ✓ Lat=-6.0487, Long=106.0615265
[4/50] Honda Anugerah Bantul
  ✓ Lat=-7.8090155, Long=110.3244686
[5/50] Honda Maju Pd Gede
  ✓ Lat=-6.284764, Long=106.906087
[6/50] Honda Thamrin Jambi
  ✓ Lat=-1.6210389, Long=103.6305954
[7/50] Honda IBRM Subang
  ✓ Lat=-6.5456099, Long=107.7783154
[8/50] Honda Kumala Cikampek
  ✓ Lat=-6.3954019, Long=107.4316914
[9/50] Honda LPPM Kuningan
  ✓ Lat=-6.9705702, Long=108.5080459
[10/50] Honda Perdana Soreang
  ✓ Lat=-7.0169646, Long=107.5465091

💾 Progress: 9 sukses, 1 gagal

[11/50] Honda Manunggal Brebes
  ✓ Lat=-6.8764692, Long=109.0664674
[12/50] Honda Mandalasena Blitar
  ✓ Lat=-8.0809294, Long=112.1965183
[13/50] Honda Prisma HR Muhammad
  ✓ Lat=-7.2860927, Long=112.7025833
[14/50] Honda Trio Pangkalan Bun
  ✓ Lat=-2.6679451, Long=111.6507279
[15/50] Honda Trio Sampit
  ✓ Lat=-2.5481482, Long=

In [46]:
# --- Tutup WebDriver ---
driver.quit()
print("WebDriver telah ditutup")


WebDriver telah ditutup


In [47]:
# --- Tambahkan Kolom Alamat_Map dan Kota ---
# Pastikan DataFrame df sudah ada dan sudah ada kolom Latitude dan Longitude
if 'df' not in locals() and 'df' not in globals():
    # Coba load dari Excel jika ada
    try:
        df = pd.read_excel('honda_dealers_indonesia.xlsx')
        print(f"DataFrame dimuat dari Excel: {len(df)} dealer")
    except:
        raise ValueError("DataFrame 'df' tidak ditemukan. Jalankan Cell 2, 3, dan 8 terlebih dahulu.")

# Tambahkan kolom baru jika belum ada
if 'Alamat_Map' not in df.columns:
    df['Alamat_Map'] = ''
if 'Kota' not in df.columns:
    df['Kota'] = ''

print(f"Kolom baru ditambahkan: Alamat_Map dan Kota")
print(f"Total dealer: {len(df)}")
print(f"\nPreview struktur DataFrame:")
print(df[['Dealer', 'Alamat', 'Latitude', 'Longitude', 'Alamat_Map', 'Kota']].head())


Kolom baru ditambahkan: Alamat_Map dan Kota
Total dealer: 192

Preview struktur DataFrame:
                  Dealer                         Alamat    Latitude  \
0  Honda Bintang Tabanan                                 -8.5530323   
1     Honda Cokroaminoto       Jl. Cokroaminoto No. 168         N/A   
2   Honda Denpasar Agung         Jl. Hayam Wuruk No. 40    -8.65731   
3     Honda Dewata Motor        Jl. Imam Bonjol No. 104    -8.66767   
4      Honda Arta Cikupa  Jl. Raya Serang KM 14, Cikupa    -6.22295   

     Longitude Alamat_Map Kota  
0  115.1362066                  
1          N/A                  
2    115.22685                  
3    115.20567                  
4      106.529                  


In [48]:
# --- Setup Selenium WebDriver untuk Scraping Alamat ---
from selenium.webdriver.chrome.options import Options

# Pastikan driver belum ada atau sudah ditutup
try:
    if 'driver' in locals() or 'driver' in globals():
        try:
            driver.quit()
        except:
            pass
except:
    pass

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=chrome_options)
print("WebDriver Chrome sudah diinisialisasi untuk scraping alamat")


WebDriver Chrome sudah diinisialisasi untuk scraping alamat


In [49]:
# --- Fungsi untuk extract alamat dan kota dari Google Maps ---
def extract_address_from_google_maps(dealer_name, lat=None, lon=None, alamat=""):
    """
    Extract alamat lengkap dan kota dari Google Maps
    Bisa menggunakan koordinat atau search dengan nama dealer
    """
    try:
        # Jika ada koordinat, gunakan untuk direct lookup
        if lat and lon and lat != 'N/A' and lon != 'N/A':
            try:
                lat_float = float(lat)
                lon_float = float(lon)
                maps_url = f"https://www.google.com/maps/@{lat_float},{lon_float},15z"
            except:
                # Jika konversi gagal, gunakan search
                query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
                maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        else:
            # Gunakan search jika tidak ada koordinat
            query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
            maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        
        driver.get(maps_url)
        time.sleep(4)
        
        # Klik hasil pertama jika belum ada detail panel
        try:
            first_result = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "div[role='article']:first-of-type, a[data-value='Directions']"))
            )
            first_result.click()
            time.sleep(3)
        except:
            # Jika sudah di detail page, lanjutkan
            pass
        
        # Coba berbagai selector untuk mendapatkan alamat
        address_text = ""
        city_text = ""
        
        # Method 1: Cari di panel info (sidebar)
        try:
            # Cari button "Save" atau area info panel
            address_elements = driver.find_elements(By.CSS_SELECTOR, 
                "button[data-item-id='address'], "
                "[data-value='Address'], "
                "button[jsaction*='address'], "
                ".Io6YTe, "
                "[data-value='Direction']")
            
            for elem in address_elements:
                text = elem.text.strip()
                if text and len(text) > 10:  # Alamat biasanya lebih dari 10 karakter
                    address_text = text
                    break
        except:
            pass
        
        # Method 2: Cari di hasil pencarian atau detail panel
        if not address_text:
            try:
                # Cari text yang mengandung indikator alamat
                all_text = driver.find_element(By.TAG_NAME, "body").text
                # Cari pattern seperti "Jl.", "Jalan", atau angka yang diikuti nama jalan
                import re
                address_patterns = [
                    r'(Jl\.?[^\\n]{20,200})',
                    r'(Jalan[^\\n]{20,200})',
                    r'([A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*\\s+No\\.?\\s+\\d+[^\\n]{10,150})'
                ]
                
                for pattern in address_patterns:
                    matches = re.findall(pattern, all_text)
                    if matches:
                        address_text = matches[0][:200]  # Ambil max 200 karakter
                        break
            except:
                pass
        
        # Method 3: Gunakan URL dengan koordinat untuk reverse geocoding manual
        if not address_text and lat and lon and lat != 'N/A' and lon != 'N/A':
            try:
                # Coba ambil dari structured data di page
                import json
                page_source = driver.page_source
                
                # Cari JSON-LD atau structured data
                if 'structured data' in page_source.lower():
                    # Extract dari meta tags atau JSON
                    pass
            except:
                pass
        
        # Extract kota dari alamat
        if address_text:
            # Kota biasanya di akhir alamat, setelah koma terakhir atau sebelum "Indonesia"
            parts = address_text.split(',')
            if len(parts) > 1:
                # Ambil 2-3 bagian terakhir (biasanya berisi kota, provinsi, atau kode pos)
                city_candidates = [p.strip() for p in parts[-3:]]
                
                # Filter untuk mendapatkan kota (biasanya tidak terlalu panjang, tidak mengandung angka kode pos)
                for candidate in reversed(city_candidates):
                    if candidate and len(candidate) < 50 and not candidate.replace(' ', '').isdigit():
                        # Skip jika mengandung kata-kata yang bukan nama kota
                        skip_words = ['Indonesia', 'Indonesia', 'Direction', 'Save', 'Share']
                        if not any(word.lower() in candidate.lower() for word in skip_words):
                            city_text = candidate
                            break
        
        # Fallback: Jika tidak dapat alamat, set kosong
        if not address_text:
            address_text = ""
        if not city_text:
            city_text = ""
            
        return address_text, city_text
        
    except Exception as e:
        print(f"Error extracting address for {dealer_name}: {str(e)}")
        return "", ""


In [50]:
# --- Fungsi yang lebih robust untuk extract alamat dari Google Maps ---
# Versi yang lebih baik dengan multiple fallback methods
def extract_address_from_google_maps_v2(dealer_name, lat=None, lon=None, alamat=""):
    """
    Extract alamat lengkap dan kota dari Google Maps dengan multiple methods
    """
    try:
        # Buat URL Google Maps
        if lat and lon and lat != 'N/A' and lon != 'N/A':
            try:
                lat_float = float(lat)
                lon_float = float(lon)
                maps_url = f"https://www.google.com/maps/place/?q=@{lat_float},{lon_float}"
            except:
                query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
                maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        else:
            query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
            maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        
        driver.get(maps_url)
        time.sleep(5)
        
        address_text = ""
        city_text = ""
        
        # Method 1: Klik hasil pertama dan ambil dari info panel
        try:
            # Tunggu search results
            first_result = WebDriverWait(driver, 8).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 
                    "div[role='article']:first-of-type, "
                    "a[data-value='Directions'], "
                    "[jsaction*='mouseover']:first-of-type"))
            )
            first_result.click()
            time.sleep(4)
        except:
            pass
        
        # Method 2: Cari di info panel menggunakan berbagai selector
        selectors_to_try = [
            "div.MngOvd span.DkEaL",
            "button[data-section-id='194'] .DkEaL",
            "span.DkEaL",
            "div.LCF4w span.DkEaL",
            "button[data-item-id='address']",
            "[data-value='Address']",
            ".Io6YTe.fontBodyMedium",
            "button[jsaction*='address']",
            "[class*='Io6YTe']",
            "[class*='fontBodyMedium']",
            "div.qbxhc",
            "span.LrzXr",
        ]
        
        for selector in selectors_to_try:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                for elem in elements:
                    text = elem.text.strip()
                    if text and len(text) > 15 and ('Jl' in text or 'Jalan' in text or ',' in text):
                        address_text = text
                        break
                if address_text:
                    break
            except:
                continue
        
        # Method 3: Ambil dari text content yang visible
        if not address_text:
            try:
                # Dapatkan semua text yang visible
                body = driver.find_element(By.TAG_NAME, "body")
                all_text = body.text
                
                # Cari pattern alamat Indonesia
                import re
                # Pattern: Jl. atau Jalan diikuti teks dan koma, kemudian kota
                pattern = r'(Jl\.?\s+[^,\\n]{10,100}(?:,\\s*[^,\\n]{5,50}){1,4})'
                matches = re.findall(pattern, all_text)
                if matches:
                    address_text = matches[0][:200]
            except:
                pass
        
        # Extract kota dari alamat
        if address_text:
            cleaned = address_text.strip()
            # Hilangkan Plus Code di awal jika ada (contoh: "6953+VCF ...")
            first_space = cleaned.find(' ')
            if '+' in cleaned.split(' ')[0] and first_space != -1:
                cleaned = cleaned[first_space+1:].strip()
            
            parts = [p.strip() for p in cleaned.split(',') if p.strip()]
            
            # Prioritaskan bagian yang mengandung 'Kota' atau 'Kabupaten'
            for part in parts:
                if part.lower().startswith('kota ') or part.lower().startswith('kabupaten '):
                    city_text = part
                    break
            
            # Jika belum ada, ambil kandidat dari 2-3 bagian terakhir
            if not city_text:
                for part in reversed(parts[-3:]):
                    if (5 < len(part) < 40 and
                        'indonesia' not in part.lower() and
                        not part.replace(' ', '').isdigit()):
                        city_text = part
                        break
        
        return address_text[:300] if address_text else "", city_text[:50] if city_text else ""
        
    except Exception as e:
        print(f"  Error: {str(e)[:100]}")
        return "", ""


In [51]:
# --- Loop untuk scraping Alamat_Map dan Kota untuk semua dealer ---
import pandas as pd
import time

# Filter dealer yang belum punya Alamat_Map atau Kota
mask = (df['Alamat_Map'] == '') | (df['Alamat_Map'].isna()) | (df['Kota'] == '') | (df['Kota'].isna())
dealers_to_scrape_address = df[mask].copy()

print(f"Total dealer: {len(df)}")
print(f"Perlu di-scrape alamat: {len(dealers_to_scrape_address)} | Sudah ada alamat: {len(df[~mask])}")

if len(dealers_to_scrape_address) > 0:
    success_count = failed_count = 0
    total = len(dealers_to_scrape_address)
    
    print(f"\n{'='*80}")
    print(f"MULAI SCRAPING ALAMAT & KOTA DARI GOOGLE MAPS - {total} DEALER")
    print(f"{'='*80}\n")
    
    for i, (idx, row) in enumerate(dealers_to_scrape_address.iterrows(), 1):
        dealer_name = row['Dealer']
        alamat = row['Alamat'] if pd.notna(row['Alamat']) else ""
        lat = row['Latitude'] if pd.notna(row['Latitude']) else None
        lon = row['Longitude'] if pd.notna(row['Longitude']) else None
        
        print(f"[{i}/{total}] {dealer_name}")
        
        # Skip jika lat/lon adalah 'N/A'
        if lat == 'N/A' or lon == 'N/A':
            lat, lon = None, None
        
        alamat_map, kota = extract_address_from_google_maps_v2(dealer_name, lat, lon, alamat)
        
        if alamat_map:
            df.at[idx, 'Alamat_Map'] = alamat_map
            success_count += 1
            print(f"  ✓ Alamat: {alamat_map[:60]}...")
        else:
            failed_count += 1
            print(f"  ✗ Gagal mendapatkan alamat")
        
        if kota:
            df.at[idx, 'Kota'] = kota
            print(f"  ✓ Kota: {kota}")
        else:
            print(f"  - Kota tidak ditemukan")
        
        time.sleep(3)  # Delay untuk menghindari rate limiting
        
        # Save setiap 10 dealer
        if i % 10 == 0:
            df.to_excel('honda_dealers_indonesia.xlsx', index=False)
            print(f"\n💾 Progress: {success_count} sukses, {failed_count} gagal (Alamat)\n")
    
    print(f"\n{'='*80}")
    print(f"SCRAPING ALAMAT SELESAI!")
    print(f"Sukses: {success_count} | Gagal: {failed_count} | Rate: {(success_count/total*100):.1f}%")
    print(f"{'='*80}")
    
    # Final save
    df.to_excel('honda_dealers_indonesia.xlsx', index=False)
    
    # Statistics
    mask_final = (df['Alamat_Map'] == '') | (df['Alamat_Map'].isna())
    print(f"\n📊 Dealer dengan Alamat_Map: {len(df[~mask_final])} | Tanpa Alamat_Map: {len(df[mask_final])}")
    
    mask_kota = (df['Kota'] == '') | (df['Kota'].isna())
    print(f"📊 Dealer dengan Kota: {len(df[~mask_kota])} | Tanpa Kota: {len(df[mask_kota])}")
    
    print(f"\n📋 Preview hasil:")
    print(df[['Dealer', 'Alamat', 'Alamat_Map', 'Kota']].head(10))
else:
    print("Semua dealer sudah memiliki Alamat_Map dan Kota")


Total dealer: 192
Perlu di-scrape alamat: 192 | Sudah ada alamat: 0

MULAI SCRAPING ALAMAT & KOTA DARI GOOGLE MAPS - 192 DEALER

[1/192] Honda Bintang Tabanan
  ✓ Alamat: C4WP+QFP Banjar Anyar, Kabupaten Tabanan, Bali...
  ✓ Kota: Kabupaten Tabanan
[2/192] Honda Cokroaminoto
  ✓ Alamat: 
Jl. Cokroaminoto No.62, Pemecutan Kaja, Kec. Denpasar Utar...
  ✓ Kota: Kota Denpasar
[3/192] Honda Denpasar Agung
  ✓ Alamat: 86VG+3PH Sumerta Kauh, Kota Denpasar, Bali...
  ✓ Kota: Kota Denpasar
[4/192] Honda Dewata Motor
  ✓ Alamat: 86J4+W7M Pemecutan Klod, Kota Denpasar, Bali...
  ✓ Kota: Kota Denpasar
[5/192] Honda Arta Cikupa
  ✓ Alamat: QGGH+RHH Dukuh, Kabupaten Tangerang, Banten...
  ✓ Kota: Kabupaten Tangerang
[6/192] Honda Auto Cilegon
  ✓ Alamat: X326+GJC Kalitimbang, Kota Cilegon, Banten...
  ✓ Kota: Kota Cilegon
[7/192] Honda Auto Serang
  ✓ Alamat: W44M+2CC Drangong, Kota Serang, Banten...
  ✓ Kota: Kota Serang
[8/192] Honda Autoland Ciputat
  ✓ Alamat: MPFW+JVW Cipayung, Kota Tangerang 

In [52]:
# --- Tutup WebDriver ---
try:
    driver.quit()
    print("WebDriver telah ditutup")
except:
    print("WebDriver sudah ditutup atau tidak ada")


WebDriver telah ditutup
